# 02 — Signal Validation

**目标：** 在交给 Agent 自动化之前，手动验证 `generate_scores()` 输出的信号质量。

```
你（ipynb 逐 Cell 验证）→ 满意 → Agent 自动化
```

每一步检查 score 分布、IC 和排序稳定性。确认之后再进入 03_topk_spread_research.ipynb。

In [ ]:
import sys, warnings, pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# ── Core interface (the point of this notebook) ──────────────
from src.core.signals import generate_scores, generate_scores_panel
from src.core.metrics import compute_ic_series
from src.common.qlib_init import build_qlib_init_cfg, safe_qlib_init

print('✅ src.core interfaces imported')

## 0. 配置 — 修改这里

In [ ]:
MARKET      = 'us'
MODEL_TAG   = 'us_absret'
TEST_START  = '2025-01-01'
TEST_END    = '2026-06-18'
SYMBOLS     = ['AAPL','NVDA','MSFT','GOOGL','AMZN','META','TSLA','AVGO','COST','NFLX']

safe_qlib_init(build_qlib_init_cfg(None, market=MARKET))
from qlib.data import D
from qlib.contrib.data.loader import Alpha158DL
print(f'Market: {MARKET.upper()}  Symbols: {len(SYMBOLS)}')

## 1. 加载已训练模型

In [ ]:
import json
from src.assistant.model_registry_index import ModelRegistryIndex
from src.assistant.metadata_db import resolve_metadata_db_path

db_path = resolve_metadata_db_path(ROOT)
reg = ModelRegistryIndex(db_path=db_path)
entries = reg.list_entries()

# 取最新的 us_absret 模型
candidates = [e for e in entries if e.get('tag') == MODEL_TAG]
assert candidates, f'No model with tag={MODEL_TAG}. Run end_to_end_training_pipeline first.'
latest = sorted(candidates, key=lambda e: e.get('created_at',''))[-1]
model_path = Path(latest['path'])
print(f'Loading: {model_path}')

with open(model_path, 'rb') as f:
    booster = pickle.load(f)
print(f'✅ Model loaded  best_iteration={getattr(booster, "best_iteration", "N/A")}')
print(f'   Registered: {latest["created_at"]}  stage={latest.get("stage","?")}')

## 2. 加载特征面板

In [ ]:
import time

alpha_exprs, _ = Alpha158DL.get_feature_config({'kbar': {}, 'price': {'windows': [0], 'feature': ['OPEN','HIGH','LOW','VWAP']}, 'rolling': {}})
extras = ['$close/Ref($close,5)-1','$close/Ref($close,10)-1','$close/Ref($close,20)-1','Std($close,10)','$volume/Ref($volume,10)-1']
all_exprs = list(alpha_exprs) + extras

t0 = time.perf_counter()
X_test = D.features(SYMBOLS, all_exprs, start_time=TEST_START, end_time=TEST_END)
X_test = X_test.fillna(0.0).replace([float('inf'), float('-inf')], 0.0)
print(f'Features loaded in {time.perf_counter()-t0:.1f}s: {X_test.shape}')

## 3. 单截面验证 — 调用 `generate_scores()`

In [ ]:
# 取最新一天的截面数据
dates = sorted(X_test.index.get_level_values(0 if X_test.index.names[0]=='datetime' else 1).unique())
latest_date = dates[-1]

if X_test.index.names[0] == 'datetime':
    X_today = X_test.xs(latest_date, level=0)
else:
    X_today = X_test.xs(latest_date, level='datetime')

# ── 调用 src.core 接口 ─────────────────────────────────────
scores = generate_scores(booster, X_today)

print(f'Date: {latest_date}  |  Stocks: {len(scores)}')
print(f'Score range: [{scores.min():.4f}, {scores.max():.4f}]  mean={scores.mean():.4f}')
print()
print('=== Top 5 (Long candidates) ===')
print(scores.sort_values(ascending=False).head())
print()
print('=== Bottom 5 (Short candidates) ===')
print(scores.sort_values(ascending=True).head())

## 4. Score 分布检查

In [ ]:
# ── 全面板分数 ─────────────────────────────────────────────
score_panel = generate_scores_panel(booster, X_test)
all_scores = score_panel['score']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# 分布
axes[0].hist(all_scores, bins=80, color='#6366f1', alpha=0.8, edgecolor='white')
axes[0].axvline(0, color='red', ls='--', lw=1, label='zero')
axes[0].axvline(all_scores.mean(), color='green', ls='--', lw=1, label=f'mean={all_scores.mean():.4f}')
axes[0].legend(fontsize=8); axes[0].set_title('Score Distribution (All Dates)')

# 按日期的 score 均值稳定性
if score_panel.index.names[0] == 'datetime':
    daily_mean = score_panel.groupby(level=0)['score'].mean()
else:
    daily_mean = score_panel.groupby(level='datetime')['score'].mean()
daily_mean.plot(ax=axes[1], color='#10b981', lw=1)
axes[1].axhline(0, color='gray', ls=':', lw=0.8)
axes[1].set_title('Daily Mean Score (should be ~stable)')

# 单日截面排序
scores.sort_values(ascending=False).plot(
    kind='bar', ax=axes[2], color='#f59e0b', alpha=0.8
)
axes[2].axhline(0, color='red', ls='--', lw=1)
axes[2].set_title(f'Cross-section Ranking ({latest_date})')
axes[2].tick_params(axis='x', labelsize=7)

plt.tight_layout(); plt.show()

# ── 检查点：问自己 ─────────────────────────────────────────
print('Sanity checks:')
print(f'  1. 分布是否接近正态（不是全正/全负）？ positive_pct={float((all_scores>0).mean()):.1%}')
print(f'  2. daily mean 是否稳定（无明显漂移）？ std_of_daily_mean={float(daily_mean.std()):.6f}')
print(f'  3. 截面是否有明显分层？ top_minus_bot={float(scores.max()-scores.min()):.4f}')

## 5. IC 验证 — 调用 `compute_ic_series()`

In [ ]:
y_raw = D.features(SYMBOLS, ['Ref($close, -10)/Ref($close, -1)-1'],
                   start_time=TEST_START, end_time=TEST_END)
return_panel = y_raw.rename(columns={y_raw.columns[0]: 'return'})

# ── 调用 src.core 接口 ─────────────────────────────────────
ic = compute_ic_series(score_panel, return_panel)

print('=== IC Summary ===')
print(f'  IC Mean:    {ic["ic_mean"]:.4f}')
print(f'  IC Std:     {ic["ic_std"]:.4f}')
print(f'  IC IR:      {ic["ic_ir"]:.4f}')
print(f'  IC Pos%:    {ic["ic_pos_pct"]:.1%}')
print(f'  Valid days: {ic["n_days"]}')

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
ic['ic_series'].plot(ax=axes[0], color='#6366f1', lw=1, alpha=0.8)
axes[0].axhline(0, color='gray', ls=':', lw=0.8)
axes[0].axhline(ic['ic_mean'], color='red', ls='--', lw=1, label=f'mean={ic["ic_mean"]:.4f}')
axes[0].legend(fontsize=8); axes[0].set_title('Daily IC Series')

axes[1].hist(ic['ic_series'], bins=40, color='#8b5cf6', alpha=0.8, edgecolor='white')
axes[1].axvline(0, color='red', ls='--', lw=1)
axes[1].axvline(ic['ic_mean'], color='green', ls='--', lw=1)
axes[1].set_title('IC Distribution')
plt.tight_layout(); plt.show()

# ── 判断标准 (仅供参考) ────────────────────────────────────
status = '✅ GOOD' if ic['ic_mean'] > 0.02 and ic['ic_ir'] > 0.3 else '⚠️ WEAK'
print(f'Signal quality: {status}  (IC>0.02 and IR>0.3 as threshold)')

## 6. 结论

在这里记录你的观察结论，满意后进入 `03_topk_spread_research.ipynb`：

- [ ] Score 分布正常（正负均衡，无明显 outlier）
- [ ] daily mean 稳定（无大幅漂移）
- [ ] IC Mean > 0 且 IC IR > 0.3
- [ ] IC Pos% > 50%